# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [142]:
# Load the libraries as required.
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os
import pickle
from glob import glob
import shap

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, log_loss, cohen_kappa_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance


from sklearn.preprocessing import (
    Binarizer,
    KBinsDiscretizer,
    LabelBinarizer,
    LabelEncoder,
    MultiLabelBinarizer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MaxAbsScaler,
    MinMaxScaler,
    Normalizer,
    RobustScaler,
    FunctionTransformer,
    KernelCenterer,
    PolynomialFeatures,
    PowerTransformer,
    QuantileTransformer,
    SplineTransformer
)


In [143]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('C:/Users/aakav/dsi/production/05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


In [144]:
print(fires_dt)

     coord_x  coord_y month  day  ffmc    dmc     dc   isi  temp  rh  wind  \
0          7        5   mar  fri  86.2   26.2   94.3   5.1   8.2  51   6.7   
1          7        4   oct  tue  90.6   35.4  669.1   6.7  18.0  33   0.9   
2          7        4   oct  sat  90.6   43.7  686.9   6.7  14.6  33   1.3   
3          8        6   mar  fri  91.7   33.3   77.5   9.0   8.3  97   4.0   
4          8        6   mar  sun  89.3   51.3  102.2   9.6  11.4  99   1.8   
..       ...      ...   ...  ...   ...    ...    ...   ...   ...  ..   ...   
512        4        3   aug  sun  81.6   56.7  665.6   1.9  27.8  32   2.7   
513        2        4   aug  sun  81.6   56.7  665.6   1.9  21.9  71   5.8   
514        7        4   aug  sun  81.6   56.7  665.6   1.9  21.2  70   6.7   
515        1        4   aug  sat  94.4  146.0  614.7  11.3  25.6  42   4.0   
516        6        3   nov  tue  79.5    3.0  106.7   1.1  11.8  31   4.5   

     rain   area  
0     0.0   0.00  
1     0.0   0.00  
2     

# Get X and Y

Create the features data frame and target data.

In [145]:
# Target variable
Y = fires_dt['area']

In [146]:
# Feature variables (feature df)
X = fires_dt.drop(columns=['area'])

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [147]:
# splitting numerical and categorical
numerical_cols = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_cols = ['month', 'day']

#Preproc 1
preproc1 = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numerical_cols),
    
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [148]:
preproc2 = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('power', PowerTransformer(method='yeo-johnson')),  
        ('scaler', StandardScaler())  
    ]), numerical_cols),
    
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [149]:
# Pipeline A = preproc1 + baseline

pipeline_a = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', LinearRegression())
])


results_a = cross_validate(
    pipeline_a,
    X,
    y,
    cv=5,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(results_a)



{'fit_time': array([0.02154136, 0.0259974 , 0.02500534, 0.02101231, 0.01999855]), 'score_time': array([0.01400375, 0.01699734, 0.01499343, 0.01800346, 0.01400566]), 'test_neg_mean_squared_error': array([  -757.95493915,   -455.264036  , -13472.63483258,   -669.81264013,
        -6793.33486457]), 'train_neg_mean_squared_error': array([-4699.21965533, -4749.46420594, -1672.27438733, -4675.13705883,
       -3224.74920889]), 'test_r2': array([  0.        , -22.41408941,  -0.06001734,  -0.55987385,
        -0.05663483]), 'train_r2': array([0.06214377, 0.05580141, 0.05164784, 0.05339328, 0.06221046])}


In [150]:
# Pipeline B = preproc2 + baseline
pipeline_b = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', LinearRegression())
])

results_b = cross_validate(
    pipeline_b,
    X,
    y,
    cv=5,
    scoring=['r2'],
    return_train_score=True
)

print(results_b)


{'fit_time': array([0.05852771, 0.05752444, 0.061064  , 0.05709338, 0.05855083]), 'score_time': array([0.01400065, 0.01500034, 0.01699805, 0.014992  , 0.01499319]), 'test_r2': array([  0.        , -28.4453911 ,  -0.06136456,  -0.46795905,
        -0.04885727]), 'train_r2': array([0.07061356, 0.06431836, 0.05297378, 0.0569092 , 0.06670601])}


In [151]:
# Pipeline C = preproc1 + advanced model

pipeline_c = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge(alpha=1.0))
])

results_c = cross_validate(
    pipeline_c,
    X,
    y,
    cv=5,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(results_c)


{'fit_time': array([0.02852607, 0.03000641, 0.02500033, 0.02352238, 0.02399707]), 'score_time': array([0.01999998, 0.0185318 , 0.01799536, 0.01699638, 0.0180161 ]), 'test_neg_mean_squared_error': array([  -623.0066793 ,   -363.1768876 , -13433.03534222,   -691.19487401,
        -6636.95255808]), 'train_neg_mean_squared_error': array([-4712.71900433, -4760.38057042, -1673.30788516, -4679.5509177 ,
       -3236.66047616]), 'test_r2': array([  0.        , -17.67807568,  -0.05690168,  -0.60966924,
        -0.03231114]), 'train_r2': array([0.05944961, 0.05363123, 0.05106174, 0.05249958, 0.05874654])}


In [152]:
# Pipeline D = preproc2 + advanced model

pipeline_d = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge(alpha=1.0))  
])

results_d = cross_validate(
    pipeline_d,
    X,
    y,
    cv=5,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(results_d)


{'fit_time': array([0.06714511, 0.05453253, 0.05053639, 0.05500627, 0.05200028]), 'score_time': array([0.01800394, 0.0170002 , 0.01518822, 0.01753187, 0.01853013]), 'test_neg_mean_squared_error': array([  -724.03329365,   -455.92983895, -13436.42285654,   -647.48249958,
        -6616.91112519]), 'train_neg_mean_squared_error': array([-4668.61532855, -4715.90638873, -1671.33977297, -4661.7917754 ,
       -3218.01633125]), 'test_r2': array([  0.        , -22.44833145,  -0.0571682 ,  -0.50787094,
        -0.0291939 ]), 'train_r2': array([0.06825169, 0.06247274, 0.05217787, 0.05609539, 0.06416845])}


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [153]:
pipeline_a.fit(X,Y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['month', 'day'])])),
                ('regressor', LinearRegression())])

In [154]:
pipeline_b.fit(X,Y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('power',
                                                                   PowerTransformer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['month', 'day'])])),
                ('regressor', LinearRegression())])

In [155]:
# AK Notes - no gridsearch on A & B
param_grid_c = {
    'regressor__alpha': [0.01, 0.1, 1, 10]
}

grid_search_c = GridSearchCV(
    pipeline_c,
    param_grid=param_grid_c,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search_c.fit(X, y)



GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             n_jobs=-1, param_grid={'regressor__alpha': [0.01, 0.1, 1, 10]},
             scoring='neg_mean_squared_error')

In [156]:
param_grid_d = {
    'regressor__alpha': [0.01, 0.1, 1, 10]
}

grid_search_d = GridSearchCV(
    pipeline_d,
    param_grid=param_grid_d,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search_d.fit(X, y)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('power',
                                                                                          PowerTransformer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             n_jobs=-1, param_grid={'regressor__alpha': [0.01, 0.1, 1, 10]},
             scoring='neg_mean_squared_error')

In [157]:
print(grid_search_c.best_params_)
print(grid_search_c.best_score_)


{'regressor__alpha': 10}
-4278.655355069459


In [158]:
print(grid_search_d.best_params_)
print(grid_search_d.best_score_)

{'regressor__alpha': 10}
-4273.04032451975


In [159]:
models = {
    "MLR + preproc1": pipeline_a,
    "MLR + preproc2": pipeline_b, 
    "Ridge + preproc1": grid_search_c,
    "Ridge + preproc2": grid_search_d,
}

for name, model in models.items():
    scores = cross_val_score(model, X, Y, cv=5, scoring='neg_root_mean_squared_error')
    print(f"{name} CV RMSE: {-scores.mean():.3f}")

MLR + preproc1 CV RMSE: 54.648
MLR + preproc2 CV RMSE: 55.419
Ridge + preproc1 CV RMSE: 52.253
Ridge + preproc2 CV RMSE: 52.388


# Evaluate

+ Which model has the best performance?

Pipeline C (Ridge + preproc1 CV) had the best performance, with the lowest RMSERMSE (52.253).

# Export

+ Save the best performing model to a pickle file.

In [160]:
best_model = grid_search_c
with open('best_model_forestfire_area.pkl','wb') as f:
    pickle.dump(best_model, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [164]:
best_pipeline = best_model.best_estimator_  # best_model = grid_search_c


X_transformed = best_pipeline.named_steps['preprocessing'].transform(X)
feature_names = best_pipeline.named_steps['preprocessing'].get_feature_names_out()

explainer = shap.Explainer(
    best_pipeline.named_steps['regressor'],
    X_transformed,
    feature_names=feature_names
)

shap_values = explainer(X_transformed)


obs_idx = 0
shap_df = pd.DataFrame({
    'feature': feature_names,
    'shap_value': shap_values[obs_idx].values
}).sort_values(by='shap_value', key=abs, ascending=False)

print("\n📌 SHAP values for observation 0:")
print(shap_df)

all_shap_df = pd.DataFrame(shap_values.values, columns=feature_names)
mean_abs_shap = all_shap_df.abs().mean().sort_values(ascending=False)

print(mean_abs_shap)


📌 SHAP values for observation 0:
           feature  shap_value
4          num__dc   16.304064
6        num__temp  -11.561206
3         num__dmc   -9.289669
17  cat__month_mar   -8.275212
22    cat__day_fri   -5.726685
8        num__wind    5.254743
21  cat__month_sep   -5.090572
0     num__coord_x    3.842517
24    cat__day_sat   -1.846302
5         num__isi    1.644260
7          num__rh   -0.888319
11  cat__month_aug    0.392438
25    cat__day_sun    0.390168
1     num__coord_y    0.387534
28    cat__day_wed    0.312411
16  cat__month_jun    0.262789
15  cat__month_jul    0.169607
23    cat__day_mon    0.152580
20  cat__month_oct   -0.128492
26    cat__day_thu   -0.099360
12  cat__month_dec   -0.058997
9        num__rain    0.049043
27    cat__day_tue   -0.044391
2        num__ffmc   -0.031847
10  cat__month_apr    0.020143
13  cat__month_feb    0.004765
19  cat__month_nov   -0.000000
14  cat__month_jan    0.000000
18  cat__month_may    0.000000
num__dc           6.611147
cat__mont

Num_raim would be be the feauture I'd remove, because it has the lowest importance according to our Sharp value. I would test this by running the entire process without num_rain and see if the RMSE changes! I can also review the adjusted R^2 values to see if there is a similar pattern with and without num_rain.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.